### 1.1. `pd.Series` и `pd.DataFrame`: столбец и таблица

Pandas удобен для таблиц с названиями столбцов, неодинаковыми типами данных и пропусками. `Series` — одномерные значения с индексом. `DataFrame` — несколько Series с общим индексом. Импорт: `import pandas as pd`.

**Как записывать**

```python
s = pd.Series([10, 20], index=["a", "b"], name="minutes")
df = pd.DataFrame({"name": ["tea", "pie"], "minutes": [5, 40]})
df["minutes"]
df[["name", "minutes"]]
```

**Параметры и свойства**

- `data` — значения Series или данные таблицы: словарь списков, массив, список словарей.
- `index` — метки строк; если не указать, используется 0…N−1.
- `columns` у DataFrame задаёт названия/выбор и порядок столбцов.
- `dtype` задаёт тип при создании, когда это применимо.
- `.shape` — строки и столбцы; `.columns` — названия; `.index` — метки строк.
- `.dtypes` — тип каждого столбца; `.head(n=5)` — первые n строк.

**Как это работает**

Ключи словаря становятся названиями столбцов. Списки одного столбца должны иметь согласованную длину. В `df["minutes"]` выбирается один столбец как Series; в `df[["minutes"]]` список названий возвращает DataFrame. Столбец `id` не превращается автоматически в индекс.

**Частая ошибка:** `df[1]` ищет столбец с названием 1, а не строку в позиции 1.

**Пример**

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 12)

toy = pd.DataFrame({'id': [101, 102, 103], 'name': ['tea', 'pie', 'soup'],
                    'minutes': [5, 40, 15], 'n_ingredients': [2, 7, 4]}, index=['a', 'b', 'c'])
display(toy)
print(toy.shape)
print(toy.dtypes)
print(type(toy['minutes']).__name__, type(toy[['minutes']]).__name__)

,id,name,minutes,n_ingredients
a,101,tea,5,2
b,102,pie,40,7
c,103,soup,15,4


(3, 4)
id               int64
name               str
minutes          int64
n_ingredients    int64
dtype: object
Series DataFrame


### Тренировка

Выбери `name` как Series, а `name` и `minutes` — как DataFrame.

<details><summary>Подсказка — открой после своей попытки</summary>

Для нескольких столбцов нужен список внутри квадратных скобок.

</details>

In [2]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [3]:
display(toy['name'])
display(toy[['name', 'minutes']])

a     tea
b     pie
c    soup
Name: name, dtype: str

,name,minutes
a,tea,5
b,pie,40
c,soup,15


**Что проверить и объяснить:** Объясни, почему `toy[1]` не выбирает вторую строку.

### 1.2. `.loc` и `.iloc`: выбор и присваивание

Используй `.loc`, когда знаешь метку строки и имя столбца; `.iloc`, когда нужны позиции. Это позволяет однозначно выбрать значения даже с нестандартным индексом.

**Как записывать**

```python
df.loc["a", "minutes"]
df.loc["a":"b", ["name", "minutes"]]
df.iloc[0:2, 0:2]
df.loc[df["minutes"] <= 20, "minutes"] = 21
```

**Параметры и свойства**

- Первый аргумент в квадратных скобках выбирает строки, второй — столбцы.
- `.loc` принимает метку, список меток, срез меток или булеву маску.
- `.iloc` принимает целочисленную позицию, список позиций или позиционный срез.
- Правая граница среза `.loc` включается, `.iloc` — нет.
- `&`, `|`, `~` комбинируют/отрицают маски; каждый сравнительный блок заключай в скобки.

**Как это работает**

Одна строка/столбец обычно возвращается как Series, несколько — как DataFrame. Для изменения используй один вызов `.loc[mask,column] = value`. Это связывает отбор и запись с исходной таблицей. Если изменения должны быть независимыми, сначала используй `.copy()`.

**Частая ошибка:** Цепное присваивание `df[mask]["minutes"] = ...` не является надёжным способом менять исходную таблицу.

**Пример**

In [4]:
display(toy.loc['a':'b', ['name', 'minutes']])
display(toy.iloc[:2, :2])
display(toy.loc[toy['minutes'] <= 15])

,name,minutes
a,tea,5
b,pie,40


,id,name
a,101,tea
b,102,pie


,id,name,minutes,n_ingredients
a,101,tea,5,2
c,103,soup,15,4


### Тренировка

Оставь рецепты с минутами ≤20 и ингредиентами ≤5. На отдельной копии увеличь их минуты на 1.

<details><summary>Подсказка — открой после своей попытки</summary>

Маска: `(условие1) & (условие2)`. Изменяй `copy.loc[маска, "minutes"]`.

</details>

In [5]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [6]:
toy_copy = toy.copy()
toy_mask = (toy_copy['minutes'] <= 20) & (toy_copy['n_ingredients'] <= 5)
toy_copy.loc[toy_mask, 'minutes'] += 1
display(toy_copy.loc[toy_mask])
assert toy.loc['a', 'minutes'] == 5

,id,name,minutes,n_ingredients
a,101,tea,6,2
c,103,soup,16,4


**Что проверить и объяснить:** Цепное присваивание `df[маска]["столбец"] = ...` не используй. В pandas 3 действует Copy-on-Write.

### 1.3. `isna`, `dropna`, `fillna` и средние

Пропуски означают неизвестные значения. Их нужно сначала посчитать, затем решить, оставить, удалить или заполнить. Они не равны нулю.

**Как записывать**

```python
df.isna().sum()
df.isna().mean()
df.isna().any(axis=1).mean()
df.dropna(subset=["minutes"])
df.fillna({"minutes": 10})
```

**Параметры и свойства**

- `isna()` возвращает булеву таблицу пропусков; `notna()` — заполненных значений.
- `any(axis=1)` — хотя бы один True в строке; `all(axis=1)` — все True.
- `dropna(axis=0, how="any", subset=...)` удаляет строки; `axis=1` — столбцы, `how="all"` — только полностью пустые.
- `fillna(value)` принимает число или словарь заполнений по столбцам.
- `mean(axis=0, skipna=True)` по умолчанию считает средние по столбцам без пропусков.
- Для доли True используй `.mean()`, для количества — `.sum()`.

**Как это работает**

Доля строк с пропусками — среднее маски, полученной после `any(axis=1)`. Средние числовых признаков считаются по заполненным значениям; числовой ID усреднять бессмысленно. Сумма пропусков по столбцам может считать одну строку несколько раз.

**Частая ошибка:** Не заполняй всё нулём автоматически. При подготовке модели параметры заполнения рассчитываются только по train.

**Пример**

In [7]:
missing_demo = pd.DataFrame({'minutes': [5., np.nan, 15.], 'steps': [2., 4., np.nan]})
display(missing_demo.isna())
print('Пропуски по столбцам:', missing_demo.isna().sum().to_dict())
print('Доля строк с пропусками:', missing_demo.isna().any(axis=1).mean())
print('Средние:', missing_demo.mean().to_dict())

,minutes,steps
0,False,False
1,True,False
2,False,True


Пропуски по столбцам: {'minutes': 1, 'steps': 1}
Доля строк с пропусками: 0.6666666666666666
Средние: {'minutes': 10.0, 'steps': 3.0}


### Тренировка

Посчитай долю пропусков в каждом столбце и долю строк без пропусков.

<details><summary>Подсказка — открой после своей попытки</summary>

Для доли по столбцам используй `.isna().mean()`, для полностью заполненных строк — отрицание `.any(axis=1)`.

</details>

In [8]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [9]:
display(missing_demo.isna().mean())
print((~missing_demo.isna().any(axis=1)).mean())

minutes    0.333333
steps      0.333333
dtype: float64

0.3333333333333333


**Что проверить и объяснить:** Пропуск не равен нулю. Не заполняй неизвестное число шагов нулём только ради удобного среднего.

### 1.4. `to_datetime`, `.dt` и `.str`

Даты и текст требуют специальных операций. `.dt` предоставляет свойства дат, `.str` выполняет строковую операцию над всем столбцом.

**Как записывать**

```python
df["submitted"] = pd.to_datetime(df["submitted"], format="%Y-%m-%d", errors="raise")
df["submitted"].dt.year
df["name"].str.title()
df["name"].str.split().str.len()
```

**Параметры и свойства**

- `to_datetime(arg, format, errors)` преобразует значения в даты.
- `format` задаёт шаблон: `%Y` — год, `%m` — месяц, `%d` — день.
- `errors="raise"` сообщает ошибку, `"coerce"` превращает неверные даты в `NaT`.
- `.dt.year`, `.dt.month`, `.dt.day` извлекают части даты.
- `.str.len()` — длина; `.str.title()` — первые заглавные буквы; `.str.strip()` — убрать внешние пробелы.
- `.str.split(pat=None)` разбивает по пробельным символам и сворачивает их повторы.

**Как это работает**

Чтобы включить весь 2010 год, используй `< pd.Timestamp("2011-01-01")`. После `.str.split()` значения — списки слов, поэтому следующий `.str.len()` считает слова, а не символы. Пропущенный текст обычно остаётся пропущенным.

**Частая ошибка:** `split(" ")` оставляет пустые элементы при повторных пробелах. Строка даты без преобразования не даёт доступ к `.dt`.

**Пример**

In [10]:
text_demo = pd.DataFrame({'name': ['green   tea', 'apple pie'], 'submitted': ['2009-12-31', '2011-01-01']})
text_demo['submitted'] = pd.to_datetime(text_demo['submitted'])
display(text_demo.assign(year=text_demo['submitted'].dt.year,
                         title=text_demo['name'].str.title(),
                         words=text_demo['name'].str.split().str.len()))

,name,submitted,year,title,words
0,green tea,2009-12-31,2009,Green Tea,2
1,apple pie,2011-01-01,2011,Apple Pie,2


### Тренировка

Выбери строки до конца 2010 года и приведи их названия к заглавным первым буквам слов.

<details><summary>Подсказка — открой после своей попытки</summary>

Граница должна включать весь 2010 год: проще условие `< "2011-01-01"`.

</details>

In [11]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [12]:
display(text_demo.loc[text_demo['submitted'] < '2011-01-01'].assign(name=lambda d: d['name'].str.title()))

,name,submitted
0,Green Tea,2009-12-31


**Что проверить и объяснить:** Сравнение с `2010-01-01` потеряло бы почти весь 2010 год.

### 1.5. `groupby` и `merge`: сводка и связь таблиц

`groupby` нужен для ответов «сколько/в среднем в каждой группе». `merge` нужен, чтобы добавить к рецепту отзывы или сведения из другой таблицы по общему ключу.

**Как записывать**

```python
counts = reviews.groupby("recipe_id").size()
means = reviews.groupby("recipe_id")["rating"].mean()
summary = reviews.groupby("recipe_id").agg(count=("rating", "size"), mean=("rating", "mean"))
joined = recipes.merge(reviews, left_on="id", right_on="recipe_id", how="left", validate="one_to_many")
```

**Параметры и свойства**

- `groupby(by)` — имя столбца, список имён или Series с ключами.
- `.size()` считает строки, `.count()` — заполненные значения.
- `.agg()` задаёт несколько агрегатов; именованные агрегаты дают понятные имена результатов.
- `merge(right, on=...)` связывает по одинаковому имени ключа.
- `left_on`/`right_on` используются при разных названиях; `right_index=True` — ключ в индексе справа.
- `how="inner"` — совпавшие ключи, `"left"` — все строки слева, `"outer"` — все ключи.
- `validate="one_to_many"` проверяет уникальность ключа слева, `"one_to_one"` — с двух сторон.

**Как это работает**

Один рецепт и несколько отзывов создают несколько строк после merge. Поэтому для числа отзывов сначала считают группы, а затем присоединяют полученный счётчик. При left-merge отсутствующий счётчик можно заполнить 0: это означает «нет строк отзывов», а не неизвестный рейтинг.

**Частая ошибка:** Среднее по всем отзывам и среднее рейтингов рецептов имеют разные веса. Отзыв с нулевым rating или пустым текстом всё равно существует.

**Пример**

In [13]:
toy_reviews = pd.DataFrame({'recipe_id': [101, 101, 103], 'rating': [5, 0, 4], 'review': ['great', None, 'ok']})
display(toy_reviews.groupby('recipe_id').agg(review_count=('rating', 'size'), mean_rating=('rating', 'mean')))
display(toy.merge(toy_reviews, left_on='id', right_on='recipe_id', how='left', validate='one_to_many'))

,review_count,mean_rating
recipe_id,,
101,2,2.5
103,1,4.0


,id,name,minutes,n_ingredients,recipe_id,rating,review
0,101,tea,5,2,101.0,5.0,great
1,101,tea,5,2,101.0,0.0,NaN
2,102,pie,40,7,NaN,NaN,NaN
3,103,soup,15,4,103.0,4.0,ok


### Тренировка

Найди ID рецептов без отзывов и добавь к `toy` число отзывов, включая нули.

<details><summary>Подсказка — открой после своей попытки</summary>

Отсутствие отзыва — отсутствие строки по ключу, а не рейтинг 0 или пустой текст. Счётчик: `groupby(...).size()`.

</details>

In [14]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [15]:
toy_counts = toy_reviews.groupby('recipe_id').size().rename('review_count')
toy_with_counts = toy.merge(toy_counts, left_on='id', right_index=True, how='left', validate='one_to_one')
toy_with_counts['review_count'] = toy_with_counts['review_count'].fillna(0).astype(int)
display(toy_with_counts)
assert toy_with_counts.loc[toy_with_counts['id'] == 102, 'review_count'].iloc[0] == 0

,id,name,minutes,n_ingredients,review_count
a,101,tea,5,2,2
b,102,pie,40,7,0
c,103,soup,15,4,1


**Что проверить и объяснить:** Нулевой рейтинг остаётся настоящим отзывом. При one-to-many число строк после merge может вырасти.

### 1.6. `pd.read_csv`: читать таблицу

`read_csv` читает текстовый файл в DataFrame и определяет столбцы. Он нужен для таблиц, где вместе хранятся числа, названия, текст и даты.

**Как записывать**

```python
df = pd.read_csv(path, sep=",", index_col=0, parse_dates=["submitted"])
```

**Параметры и свойства**

- `sep` — разделитель, запятая по умолчанию.
- `header=0` — первая строка содержит названия; `header=None` — названий нет.
- `names` задаёт имена вручную.
- `index_col` выбирает столбец индекса; без него Pandas создаёт новый последовательный индекс.
- `usecols` ограничивает столбцы; `dtype` задаёт типы.
- `parse_dates` преобразует перечисленные столбцы в даты.
- `na_values` добавляет обозначения пропусков; `nrows` ограничивает число строк.

**Как это работает**

Если CSV был сохранён вместе с индексом, безымянный первый столбец можно считать через index_col=0. Для обычного файла рецептов ID остаётся отдельным столбцом. После чтения проверь head, shape и dtypes.

**Частая ошибка:** Не указывай index_col=0 автоматически для каждого CSV: так можно случайно убрать первый полезный столбец.

**Пример**

In [16]:
from io import StringIO
text_csv = "name;minutes;submitted\ntea;5;2009-01-01\npie;40;2011-01-01\n"
csv_demo = pd.read_csv(StringIO(text_csv), sep=";", parse_dates=["submitted"])
display(csv_demo)
print(csv_demo.dtypes)

,name,minutes,submitted
0,tea,5,2009-01-01
1,pie,40,2011-01-01


name                    str
minutes               int64
submitted    datetime64[us]
dtype: object


### Тренировка · чтение CSV

Прочитай тот же текст, оставив только name и minutes.

In [17]:
# Твоё решение

**Эталон**

In [18]:
csv_selected = pd.read_csv(StringIO(text_csv),sep=";",usecols=["name","minutes"])
display(csv_selected)
assert csv_selected.shape == (2,2)

,name,minutes
0,tea,5
1,pie,40


In [19]:
from pathlib import Path
from IPython.display import display
# Найти исходные данные относительно текущей папки.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if all((p / folder).is_dir() for folder in ('01','02','03')))


## Лабораторная №2: все 18 исходных заданий

Одна строка `recipes` — рецепт. Одна строка `reviews` — отзыв. У рецепта может быть много отзывов. Это определяет смысл группировок и размер результата соединения. Ниже используются только файлы папки `02`.

### ЛР2 · задание 1.1

1.1 В файлах `recipes_sample.csv` и `reviews_sample.csv` находится информация об рецептах блюд и отзывах на эти рецепты соответственно. Загрузите данные из файлов в виде `pd.DataFrame` с названиями `recipes` и `reviews`. Обратите внимание на корректное считывание столбца с индексами в таблице `reviews` (безымянный столбец).

<details><summary>Подсказка — открой после своей попытки</summary>

В `reviews` безымянный первый столбец — сохранённый индекс; считай его через `index_col=0`.

</details>

In [20]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [21]:
recipes = pd.read_csv(ROOT / '02/recipes_sample.csv')
reviews = pd.read_csv(ROOT / '02/reviews_sample.csv', index_col=0)
assert recipes['id'].is_unique
assert not reviews.columns.str.startswith('Unnamed').any()
display(recipes.head())
display(reviews.head())

,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients
0,george s at the cove black bean soup,44123,90,35193,2002-10-25,NaN,an original recipe created by chef scott meska...,18.0
1,healthy for them yogurt popsicles,67664,10,91970,2003-07-26,NaN,my children and their friends ask for my homem...,NaN
2,i can t believe it s spinach,38798,30,1533,2002-08-29,NaN,"these were so go, it surprised even me.",8.0
3,italian gut busters,35173,45,22724,2002-07-27,NaN,my sister-in-law made these for us at a family...,NaN
4,love is in the air beef fondue sauces,84797,25,4470,2004-02-23,4.0,i think a fondue is a very romantic casual din...,NaN


,user_id,recipe_id,date,rating,review
370476,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...
624300,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...
187037,400708,252013,2008-01-10,4,"Very nice breakfast HH, easy to make and yummy..."
706134,2001852463,404716,2017-12-11,5,These are a favorite for the holidays and so e...
312179,95810,129396,2008-03-14,5,Excellent soup! The tomato flavor is just gre...


**Что проверить и объяснить:** Пока даты строковые. Это изменим в 2.1. Индекс отзывов не является ID рецепта.

### ЛР2 · задание 1.2

1.2 Для каждой из таблиц выведите основные параметры:
* количество точек данных (строк);
* количество столбцов;
* тип данных каждого столбца.

<details><summary>Подсказка — открой после своей попытки</summary>

Форма — `.shape`, типы — `.dtypes`.

</details>

In [22]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [23]:
print('recipes: строк', len(recipes), 'столбцов', recipes.shape[1])
display(recipes.dtypes)
print('reviews: строк', len(reviews), 'столбцов', reviews.shape[1])
display(reviews.dtypes)

recipes: строк 30000 столбцов 8


name                  str
id                  int64
minutes             int64
contributor_id      int64
submitted             str
n_steps           float64
description           str
n_ingredients     float64
dtype: object

reviews: строк 126696 столбцов 5


user_id      int64
recipe_id    int64
date           str
rating       int64
review         str
dtype: object

**Что проверить и объяснить:** В рецептах 30000 строк и 8 столбцов, в отзывах 126696 строк и 5 столбцов.

### ЛР2 · задание 1.3

1.3 Исследуйте, в каких столбцах таблиц содержатся пропуски. Посчитайте долю строк, содержащих пропуски, в отношении к общему количеству строк.

<details><summary>Подсказка — открой после своей попытки</summary>

Посчитай пропуски по столбцам, затем `.isna().any(axis=1).mean()` для доли строк.

</details>

In [24]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [25]:
display(recipes.isna().sum().rename('Пропусков recipes'))
display(reviews.isna().sum().rename('Пропусков reviews'))
print(f'Строк с пропусками recipes: {recipes.isna().any(axis=1).mean():.2%}')
print(f'Строк с пропусками reviews: {reviews.isna().any(axis=1).mean():.2%}')

name                  0
id                    0
minutes               0
contributor_id        0
submitted             0
n_steps           11190
description         623
n_ingredients      8880
Name: Пропусков recipes, dtype: int64

user_id       0
recipe_id     0
date          0
rating        0
review       17
Name: Пропусков reviews, dtype: int64

Строк с пропусками recipes: 56.85%
Строк с пропусками reviews: 0.01%


**Что проверить и объяснить:** Сумма пропусков по столбцам не равна числу строк с пропусками: у строки может быть несколько отсутствующих значений.

### ЛР2 · задание 1.4

1.4 Рассчитайте среднее значение для каждого из числовых столбцов (где это имеет смысл).

<details><summary>Подсказка — открой после своей попытки</summary>

Среднее имеет смысл для minutes, n_steps, n_ingredients и rating. Для ID пользователей и рецептов — нет.

</details>

In [26]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [27]:
display(recipes[['minutes', 'n_steps', 'n_ingredients']].mean().rename('Среднее'))
print('Средний рейтинг:', reviews['rating'].mean())
display(recipes[['minutes', 'n_steps', 'n_ingredients']].count().rename('Заполненных значений'))

minutes          123.358133
n_steps            9.805582
n_ingredients      9.008286
Name: Среднее, dtype: float64

Средний рейтинг: 4.410802235271832


minutes          30000
n_steps          18810
n_ingredients    21120
Name: Заполненных значений, dtype: int64

**Что проверить и объяснить:** Не усредняй все числовые столбцы механически. Средние по шагам и ингредиентам используют только заполненные значения.

### ЛР2 · задание 1.5

1.5 Создайте серию из 10 случайных названий рецептов.

<details><summary>Подсказка — открой после своей попытки</summary>

Выбери столбец названий и используй `.sample(n=10, random_state=42)`.

</details>

In [28]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [29]:
random_names = recipes['name'].sample(n=10, random_state=42)
display(random_names)
assert isinstance(random_names, pd.Series) and len(random_names) == 10

2308                                    barbecued potatoes
22404                                reena s pickled beets
23397        saucy mocha pots of cream  microwave easy fix
25058                                      spaghetti salad
2664                      beef steaks with capsicum relish
8511                 crispy low carb fried chicken nuggets
5148     cheese ravioli with white wine sauce and zucchini
7790                             cowboy sourdough pancakes
11311                                 french toast muffins
19043                              old fashion egg custard
Name: name, dtype: str

**Что проверить и объяснить:** Это Series со старым индексом, а не таблица и не список.

### ЛР2 · задание 1.6

1.6 Измените индекс в таблице `reviews`, пронумеровав строки, начиная с нуля.

<details><summary>Подсказка — открой после своей попытки</summary>

Новый последовательный индекс создаёт `.reset_index(drop=True)`.

</details>

In [30]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [31]:
reviews = reviews.reset_index(drop=True)
print(reviews.index)
assert reviews.index.equals(pd.RangeIndex(len(reviews)))

RangeIndex(start=0, stop=126696, step=1)


**Что проверить и объяснить:** Без `drop=True` старый индекс превратился бы в лишний столбец.

### ЛР2 · задание 1.7

1.7 Выведите информацию о рецептах, время выполнения которых не больше 20 минут и кол-во ингредиентов в которых не больше 5.

<details><summary>Подсказка — открой после своей попытки</summary>

Маска с двумя сравнениями. Пропущенные ингредиенты не подтверждают условие.

</details>

In [32]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [33]:
simple_recipes = recipes.loc[(recipes['minutes'] <= 20) & (recipes['n_ingredients'] <= 5)]
print('Подходящих рецептов:', len(simple_recipes))
display(simple_recipes.head())

Подходящих рецептов: 2019


,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients
28,quick biscuit bread,302399,20,213909,2008-05-06,11.0,this is a wonderful quick bread to make as an ...,5.0
60,peas fit for a king or queen,303944,20,213909,2008-05-16,NaN,this recipe is so simple and the flavors are s...,5.0
90,hawaiian sunrise mimosa,100837,5,58104,2004-09-29,4.0,pineapple mimosa was changed to hawaiian sunri...,3.0
91,tasty dish s banana pudding in 2 minutes,286484,2,47892,2008-02-13,NaN,"""mmmm, i love bananas!"" a --tasty dish-- origi...",4.0
94,1 minute meatballs,11361,13,4470,2001-09-03,NaN,this is a real short cut for cooks in a hurry....,2.0


**Что проверить и объяснить:** У NumPy в ЛР1 другой файл: не требуй совпадения процентов между двумя датасетами.

### ЛР2 · задание 2.1

2.1 Преобразуйте столбец `submitted` из таблицы `recipes` в формат времени. Модифицируйте решение задачи 1.1 так, чтобы считать столбец сразу в нужном формате.

<details><summary>Подсказка — открой после своей попытки</summary>

Сначала `pd.to_datetime`; альтернативное чтение — `parse_dates=["submitted"]`.

</details>

In [34]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [35]:
recipes['submitted'] = pd.to_datetime(recipes['submitted'], errors='raise')
# Так можно изменить первоначальное чтение из задания 1.1:
recipes_with_dates = pd.read_csv(ROOT / '02/recipes_sample.csv', parse_dates=['submitted'])
assert pd.api.types.is_datetime64_any_dtype(recipes['submitted'])
assert recipes['submitted'].equals(recipes_with_dates['submitted'])
display(recipes[['id', 'submitted']].head())

,id,submitted
0,44123,2002-10-25
1,67664,2003-07-26
2,38798,2002-08-29
3,35173,2002-07-27
4,84797,2004-02-23


**Что проверить и объяснить:** Дата — не произвольная строка. Проверка `errors="raise"` не скрывает проблемы формата.

### ЛР2 · задание 2.2

2.2 Выведите информацию о рецептах, добавленных в датасет не позже 2010 года.

<details><summary>Подсказка — открой после своей попытки</summary>

«Не позже 2010 года» означает весь 2010 год включительно.

</details>

In [36]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [37]:
old_recipes = recipes.loc[recipes['submitted'] < pd.Timestamp('2011-01-01')]
print('Число рецептов:', len(old_recipes))
display(old_recipes[['id', 'name', 'submitted']].head())
assert old_recipes['submitted'].dt.year.max() <= 2010

Число рецептов: 27661


,id,name,submitted
0,44123,george s at the cove black bean soup,2002-10-25
1,67664,healthy for them yogurt popsicles,2003-07-26
2,38798,i can t believe it s spinach,2002-08-29
3,35173,italian gut busters,2002-07-27
4,84797,love is in the air beef fondue sauces,2004-02-23


**Что проверить и объяснить:** Объясни, почему граница 2011-01-01 исключается.

### ЛР2 · задание 3.1

3.1  Добавьте в таблицу `recipes` столбец `description_length`, в котором хранится длина описания рецепта из столбца `description`.

<details><summary>Подсказка — открой после своей попытки</summary>

Используй `.str.len()`; неизвестное описание оставь неизвестным.

</details>

In [38]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [39]:
recipes['description_length'] = recipes['description'].str.len()
display(recipes[['description', 'description_length']].head())
assert recipes['description_length'].isna().equals(recipes['description'].isna())

,description,description_length
0,an original recipe created by chef scott meska...,330.0
1,my children and their friends ask for my homem...,255.0
2,"these were so go, it surprised even me.",39.0
3,my sister-in-law made these for us at a family...,154.0
4,i think a fondue is a very romantic casual din...,587.0


**Что проверить и объяснить:** Пропущенное описание не обязательно пустое: длина остаётся пропуском, а не автоматически 0.

### ЛР2 · задание 3.2

3.2 Измените название каждого рецепта в таблице `recipes` таким образом, чтобы каждое слово в названии начиналось с прописной буквы.

<details><summary>Подсказка — открой после своей попытки</summary>

Векторная операция `.str.title()`.

</details>

In [40]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [41]:
recipes['name'] = recipes['name'].str.title()
display(recipes['name'].head())

0       George S At The Cove  Black Bean Soup
1          Healthy For Them  Yogurt Popsicles
2                I Can T Believe It S Spinach
3                        Italian  Gut Busters
4    Love Is In The Air  Beef Fondue   Sauces
Name: name, dtype: str

**Что проверить и объяснить:** Изменение регистров не меняет число рецептов. При повторном запуске результат тот же.

### ЛР2 · задание 3.3

3.3 Добавьте в таблицу `recipes` столбец `name_word_count`, в котором хранится количество слов из названии рецепта (считайте, что слова в названии разделяются только пробелами). Обратите внимание, что между словами может располагаться несколько пробелов подряд.

<details><summary>Подсказка — открой после своей попытки</summary>

`.str.split()` без аргумента корректно обрабатывает несколько пробелов подряд.

</details>

In [42]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [43]:
recipes['name_word_count'] = recipes['name'].str.split().str.len()
display(recipes[['name', 'name_word_count']].head())
assert pd.Series(['green   tea']).str.split().str.len().iloc[0] == 2

,name,name_word_count
0,George S At The Cove Black Bean Soup,8
1,Healthy For Them Yogurt Popsicles,5
2,I Can T Believe It S Spinach,7
3,Italian Gut Busters,3
4,Love Is In The Air Beef Fondue Sauces,8


**Что проверить и объяснить:** Почему `split(" ")` на трёх пробелах создаёт лишние пустые элементы?

### ЛР2 · задание 4.1

4.1 Посчитайте количество рецептов, представленных каждым из участников (`contributor_id`). Какой участник добавил максимальное кол-во рецептов?

<details><summary>Подсказка — открой после своей попытки</summary>

Количество строк группы — `.groupby("contributor_id").size()`. Проверь все возможные ничьи.

</details>

In [44]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [45]:
contributor_counts = recipes.groupby('contributor_id').size().sort_values(ascending=False)
display(contributor_counts.head(10).rename('Рецептов'))
leaders = contributor_counts[contributor_counts == contributor_counts.max()]
print('Максимум рецептов:', int(contributor_counts.max()))
display(leaders.rename('Все лидеры'))

contributor_id
89831     421
37449     346
37779     345
1533      186
169430    183
58104     180
80353     147
4470      141
283251    127
166642    115
Name: Рецептов, dtype: int64

Максимум рецептов: 421


contributor_id
89831    421
Name: Все лидеры, dtype: int64

**Что проверить и объяснить:** ID автора — группирующая метка. Не вычисляй его среднее.

### ЛР2 · задание 4.2

4.2 Посчитайте средний рейтинг к каждому из рецептов. Для скольких рецептов отсутствуют отзывы? Обратите внимание, что отзыв с нулевым рейтингом или не заполненным текстовым описанием не считается отсутствующим.

<details><summary>Подсказка — открой после своей попытки</summary>

Рейтинг агрегируй по recipe_id. Отсутствие отзывов проверяй по ключам, независимо от rating и review.

</details>

In [46]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [47]:
mean_ratings = reviews.groupby('recipe_id')['rating'].mean().rename('mean_rating')
no_reviews = recipes.loc[~recipes['id'].isin(reviews['recipe_id']), ['id', 'name']]
display(mean_ratings.head())
print('Рецептов без отзывов:', len(no_reviews))
display(no_reviews.head())

recipe_id
48    1.000000
55    4.750000
66    4.944444
91    4.750000
94    5.000000
Name: mean_rating, dtype: float64

Рецептов без отзывов: 1900


,id,name
26,223349,Pasta With Shrimp Eggplant
29,342620,Secret Ingredient Bbq Meatloaf
35,276594,Windy S Sweet And Sour Meatballs
49,216068,Goulashy Beef Stew For The Slow Cooker
59,306590,Old Bay Grilled Steak Fries


**Что проверить и объяснить:** Ноль в rating учитывается. Пустой текст отзыва не удаляет строку.

### ЛР2 · задание 4.3

4.3 Посчитайте количество рецептов с разбивкой по годам создания.

<details><summary>Подсказка — открой после своей попытки</summary>

Сгруппируй по `.dt.year` и посчитай размер каждой группы.

</details>

In [48]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [49]:
recipes_by_year = recipes.groupby(recipes['submitted'].dt.year).size().rename('recipe_count')
display(recipes_by_year)
assert recipes_by_year.sum() == len(recipes)

submitted
1999     275
2000     104
2001     589
2002    2644
2003    2334
        ... 
2014     139
2015      42
2016      24
2017      39
2018      24
Name: recipe_count, Length: 20, dtype: int64

**Что проверить и объяснить:** Сумма годовых количеств равна числу рецептов, поскольку даты здесь заполнены.

### ЛР2 · задание 5.1

5.1 При помощи объединения таблиц, создайте `DataFrame`, состоящий из четырех столбцов: `id`, `name`, `user_id`, `rating`. Рецепты, на которые не оставлен ни один отзыв, должны отсутствовать в полученной таблице. Подтвердите правильность работы вашего кода, выбрав рецепт, не имеющий отзывов, и попытавшись найти строку, соответствующую этому рецепту, в полученном `DataFrame`.

<details><summary>Подсказка — открой после своей попытки</summary>

Используй inner-merge по `recipes.id == reviews.recipe_id`; затем оставь ровно 4 столбца.

</details>

In [50]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [51]:
recipe_reviews = recipes[['id', 'name']].merge(
    reviews[['recipe_id', 'user_id', 'rating']], left_on='id', right_on='recipe_id',
    how='inner', validate='one_to_many')
recipe_reviews = recipe_reviews[['id', 'name', 'user_id', 'rating']]
display(recipe_reviews.head())
if len(no_reviews):
    missing_id = no_reviews['id'].iloc[0]
    print('Проверяем рецепт без отзывов:', missing_id)
    display(recipe_reviews.loc[recipe_reviews['id'] == missing_id])
    assert not recipe_reviews['id'].eq(missing_id).any()
else:
    print('В этом наборе нет рецептов без отзывов.')

,id,name,user_id,rating
0,44123,George S At The Cove Black Bean Soup,743566,5
1,44123,George S At The Cove Black Bean Soup,76503,5
2,44123,George S At The Cove Black Bean Soup,34206,5
3,67664,Healthy For Them Yogurt Popsicles,494084,5
4,67664,Healthy For Them Yogurt Popsicles,303445,5


Проверяем рецепт без отзывов: 223349


,id,name,user_id,rating


**Что проверить и объяснить:** Одна строка результата — отзыв с названием рецепта. Не жди одну строку на рецепт.

### ЛР2 · задание 5.2

5.2 При помощи объединения таблиц и группировок, создайте `DataFrame`, состоящий из трех столбцов: `recipe_id`, `name`, `review_count`, где столбец `review_count` содержит кол-во отзывов, оставленных на рецепт `recipe_id`. У рецептов, на которые не оставлен ни один отзыв, в столбце `review_count` должен быть указан 0. Подтвердите правильность работы вашего кода, выбрав рецепт, не имеющий отзывов, и найдя строку, соответствующую этому рецепту, в полученном `DataFrame`.

<details><summary>Подсказка — открой после своей попытки</summary>

Сначала сгруппируй отзывы, затем left-merge счётчика с рецептами и заполни отсутствующий счётчик нулём.

</details>

In [52]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [53]:
review_counts = reviews.groupby('recipe_id').size().rename('review_count')
recipes_review_count = recipes[['id', 'name']].rename(columns={'id': 'recipe_id'}).merge(
    review_counts, left_on='recipe_id', right_index=True, how='left', validate='one_to_one')
recipes_review_count['review_count'] = recipes_review_count['review_count'].fillna(0).astype(int)
display(recipes_review_count.head())
assert len(recipes_review_count) == len(recipes)
if len(no_reviews):
    missing_id = no_reviews['id'].iloc[0]
    missing_row = recipes_review_count.loc[recipes_review_count['recipe_id'] == missing_id]
    display(missing_row)
    assert missing_row['review_count'].iloc[0] == 0

,recipe_id,name,review_count
0,44123,George S At The Cove Black Bean Soup,3
1,67664,Healthy For Them Yogurt Popsicles,8
2,38798,I Can T Believe It S Spinach,3
3,35173,Italian Gut Busters,1
4,84797,Love Is In The Air Beef Fondue Sauces,8


,recipe_id,name,review_count
26,223349,Pasta With Shrimp Eggplant,0


**Что проверить и объяснить:** Считай отзывы через `size()`, не через `review.count()`: пустой текст всё равно отзыв.

### ЛР2 · задание 5.3.

5.3. Выясните, рецепты, добавленные в каком году, имеют наименьший средний рейтинг?

<details><summary>Подсказка — открой после своей попытки</summary>

Проверь смысл среднего: сначала средний рейтинг каждого рецепта, затем среднее этих рейтингов по году добавления рецепта.

</details>

In [54]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [55]:
recipes_rated = recipes[['id', 'submitted']].merge(mean_ratings, left_on='id', right_index=True,
                                                       how='left', validate='one_to_one')
year_ratings = recipes_rated.groupby(recipes_rated['submitted'].dt.year)['mean_rating'].agg(['mean', 'count'])
display(year_ratings)
valid_year_means = year_ratings['mean'].dropna()
worst_years = valid_year_means[valid_year_means == valid_year_means.min()]
print('Год(ы) с минимальным средним рейтингом рецептов:')
display(worst_years)
# Альтернативный смысл: равный вес каждого отзыва, а не каждого рецепта.
review_weighted = recipes[['id', 'submitted']].merge(reviews[['recipe_id', 'rating']],
    left_on='id', right_on='recipe_id', how='inner', validate='one_to_many')
display(review_weighted.groupby(review_weighted['submitted'].dt.year)['rating'].mean().rename('Среднее по отзывам'))

,mean,count
submitted,,
1999,4.058786,265
2000,4.168185,98
2001,4.245481,561
2002,4.281093,2514
2003,4.380103,2225
...,...,...
2014,4.400553,127
2015,4.064969,36
2016,3.456522,23


Год(ы) с минимальным средним рейтингом рецептов:


submitted
2017    2.707071
Name: mean, dtype: float64

submitted
1999    4.274895
2000    4.284585
2001    4.393945
2002    4.404645
2003    4.439152
          ...   
2014    4.538235
2015    4.207317
2016    3.538462
2017    2.750000
2018    3.388889
Name: Среднее по отзывам, Length: 20, dtype: float64

**Что проверить и объяснить:** Основной ответ даёт равный вес каждому оценённому рецепту; альтернативный — каждому отзыву. Рецепты без отзывов не становятся рецептами с рейтингом 0.

## Работа по ЕГЭ: все 11 заданий из PDF

Источник: `02/Простой анализ данных.pdf`, данные: `02/data.xlsx`, лист «Выполнение заданий». Это **учебный анализ исторического файла**, а не актуальные правила экзамена.

В книге перед шапкой две служебные строки, поэтому нужен `header=2`. В конце — строка «Среднее значение»; её нельзя считать учеником. Короткие ответы записаны строками `+`/`-`, развёрнутые — строками вида `1(2)0(3)`: набранные баллы и максимум в скобках.

В PDF не заданы границы перевода баллов в «отлично/хорошо/удовлетворительно». Для задания 5 явно примем **условную учебную шкалу**: ниже минимального балла — неудовлетворительно, от него до 50 — удовлетворительно, от 50 до 70 — хорошо, от 70 — отлично. Это параметры примера, не официальная шкала. Порог сдачи берём из самой книги. Для сравнения школ выберем две самые многочисленные; можно заменить их номера.

### ЕГЭ · задание 1

1. Загрузите данные в ноутбук.

<details><summary>Подсказка — открой после своей попытки</summary>

Отдели записи учеников по непустому №; имя итоговой строки не нужно угадывать.

</details>

In [56]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [57]:
exam_raw = pd.read_excel(ROOT / '02/data.xlsx', sheet_name='Выполнение заданий', header=2)
exam = exam_raw.loc[exam_raw['№'].notna()].copy()
exam['№ школы'] = exam['№ школы'].astype(int)
exam['Балл'] = pd.to_numeric(exam['Балл'], errors='raise')
assert exam['№'].is_unique and exam['Балл'].notna().all()
print('Строк до очистки:', len(exam_raw), 'Учеников:', len(exam))
display(exam.head())

Строк до очистки: 2357 Учеников: 2356


,№,№ школы,Класс,Пол,Номер варианта,Задания с кратким ответом,Задания с развёрнутым ответом,Устная часть,Первичный балл,Балл,Минимальный балл
0,1.0,124,11,Ж,341.0,------+-----,0(2)0(2)0(2)0(3)0(3)0(4)0(4),NaN,1.0,5.0,27.0
1,2.0,128,11,Ж,345.0,-+----------,0(2)0(2)0(2)0(3)0(3)0(4)0(4),NaN,1.0,5.0,27.0
2,3.0,132,11,Ж,340.0,-+----------,0(2)0(2)0(2)0(3)0(3)0(4)0(4),NaN,1.0,5.0,27.0
3,4.0,134,11,Ж,339.0,-+----------,0(2)0(2)0(2)0(3)0(3)0(4)0(4),NaN,1.0,5.0,27.0
4,5.0,136,11,Ж,348.0,--------+---,0(2)0(2)0(2)0(3)0(3)0(4)0(4),NaN,1.0,5.0,27.0


**Что проверить и объяснить:** 2356 учеников; итоговая строка исключена. Сравни с исходным Excel.

### ЕГЭ · задание 2

2. Сделайте описательную статистику полученных данных.

<details><summary>Подсказка — открой после своей попытки</summary>

Числовую статистику строй для баллов, а не для номеров учеников и школ.

</details>

In [58]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [59]:
display(exam[['Первичный балл', 'Балл', 'Минимальный балл']].describe())
display(exam.isna().sum().rename('Пропуски'))

,Первичный балл,Балл,Минимальный балл
count,2356.000000,2356.000000,2356.0
mean,10.123939,49.255093,27.0
std,3.820484,17.880420,0.0
min,1.000000,5.000000,27.0
25%,7.000000,33.000000,27.0
50%,10.000000,50.000000,27.0
75%,13.000000,68.000000,27.0
max,27.000000,96.000000,27.0


№                                   0
№ школы                             0
Класс                               0
Пол                                 0
Номер варианта                      0
Задания с кратким ответом           0
Задания с развёрнутым ответом       0
Устная часть                     2356
Первичный балл                      0
Балл                                0
Минимальный балл                    0
Name: Пропуски, dtype: int64

**Что проверить и объяснить:** Одна строка — один ученик. «Устная часть» в этом файле не заполнена и не участвует в расчётах.

### ЕГЭ · задание 3

3. Найдите процент учащихся, выполнивших работу ниже среднего.

<details><summary>Подсказка — открой после своей попытки</summary>

Средний балл считай только по очищенной таблице. Сравнение строгое.

</details>

In [60]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [61]:
mean_exam_score = exam['Балл'].mean()
below_average_pct = 100 * exam['Балл'].lt(mean_exam_score).mean()
print(f'Средний балл: {mean_exam_score:.2f}; ниже среднего: {below_average_pct:.2f}%')

Средний балл: 49.26; ниже среднего: 48.01%


**Что проверить и объяснить:** Знаменатель — 2356 учеников. Итоговая строка не добавляет наблюдение.

### ЕГЭ · задание 4

4. Найдите процент учащихся, не сдавших экзамен.

<details><summary>Подсказка — открой после своей попытки</summary>

Для каждой строки сравни Балл с её Минимальным баллом.

</details>

In [62]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [63]:
assert exam['Минимальный балл'].notna().all()
failed_exam = exam['Балл'] < exam['Минимальный балл']
print('Минимальные баллы в файле:', exam['Минимальный балл'].unique())
print(f'Не сдали: {failed_exam.sum()} ({failed_exam.mean():.2%})')

Минимальные баллы в файле: [27.]
Не сдали: 217 (9.21%)


**Что проверить и объяснить:** Балл, равный порогу, считается сдачей.

### ЕГЭ · задание 5

5. Найдите процентное соотношение учащихся, сдавших экзамен на «отлично», «хорошо», «удовлетворительно», «неудовлетворительно».

<details><summary>Подсказка — открой после своей попытки</summary>

Используй явно объявленную учебную шкалу. Порог сдачи в этом наборе один.

</details>

In [64]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [65]:
GOOD_FROM = 50
EXCELLENT_FROM = 70
assert exam['Минимальный балл'].nunique() == 1
pass_from = float(exam['Минимальный балл'].iloc[0])
assert pass_from < GOOD_FROM < EXCELLENT_FROM
grade_order = ['неудовлетворительно', 'удовлетворительно', 'хорошо', 'отлично']
exam['Учебная оценка'] = pd.cut(exam['Балл'], bins=[-np.inf, pass_from, GOOD_FROM, EXCELLENT_FROM, np.inf],
                               labels=grade_order, right=False)
grade_shares = exam['Учебная оценка'].value_counts(normalize=True, sort=False).mul(100)
display(grade_shares.rename('Процент по условной шкале'))
assert np.isclose(grade_shares.sum(), 100)

Учебная оценка
неудовлетворительно     9.210526
удовлетворительно      38.794567
хорошо                 32.894737
отлично                19.100170
Name: Процент по условной шкале, dtype: float64

**Что проверить и объяснить:** Если преподаватель задаст другие границы, измени параметры и перезапусти. Эти оценки не следует выдавать за официальные.

### ЕГЭ · задание 6

6. Какое процентное соотношение юношей и девушек писало данный экзамен?

<details><summary>Подсказка — открой после своей попытки</summary>

Проверь категории и пропуски до нормализации.

</details>

In [66]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [67]:
assert exam['Пол'].notna().all()
display(exam['Пол'].value_counts().rename('Учеников'))
display(exam['Пол'].value_counts(normalize=True).mul(100).rename('Процент'))

Пол
Ж    1245
М    1111
Name: Учеников, dtype: int64

Пол
Ж    52.843803
М    47.156197
Name: Процент, dtype: float64

**Что проверить и объяснить:** Ж — 1245, М — 1111; сумма равна числу учеников.

### ЕГЭ · задание 7

7. Сколько школ принимало участие в экзамене?

<details><summary>Подсказка — открой после своей попытки</summary>

Разные номера школ: `.nunique()`.

</details>

In [68]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [69]:
print('Школ:', exam['№ школы'].nunique())
display(exam['№ школы'].value_counts().head(10).rename('Учеников'))

Школ: 17


№ школы
136    1020
135     182
127     176
133     115
128     113
123     111
132     104
137      88
121      81
125      77
Name: Учеников, dtype: int64

**Что проверить и объяснить:** Номер школы — категория, не числовой показатель для среднего.

### ЕГЭ · задание 8

8. Сколько всего заданий с кратким ответом? С развёрнутым ответом?

<details><summary>Подсказка — открой после своей попытки</summary>

B — длина строки +/-. C — число пар балл(максимум). Проверь согласованность у всех учеников.

</details>

In [70]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [71]:
import re
short_answers = exam['Задания с кратким ответом']
long_answers = exam['Задания с развёрнутым ответом']
assert short_answers.notna().all() and short_answers.str.fullmatch(r'[+-]+').all()
assert long_answers.notna().all() and long_answers.str.fullmatch(r'(?:\d+\(\d+\))+').all()
assert short_answers.str.len().nunique() == 1
long_pairs = long_answers.map(lambda value: re.findall(r'(\d+)\((\d+)\)', value))
assert long_pairs.map(len).nunique() == 1
N_B = int(short_answers.str.len().iloc[0])
N_C = len(long_pairs.iloc[0])
print('Коротких заданий:', N_B, 'Развёрнутых:', N_C)

Коротких заданий: 12 Развёрнутых: 7


**Что проверить и объяснить:** В файле 12 коротких и 7 развёрнутых заданий. Регулярное выражение здесь парсит формат, а не оценивает ученика.

### ЕГЭ · задание 9

9. Для заданий В1…Вк посчитайте процент выполненных и невыполненных заданий по каждому вопросу класса В.

<details><summary>Подсказка — открой после своей попытки</summary>

Преобразуй строки +/− в отдельные столбцы. + значит выполнено.

</details>

In [72]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [73]:
B = pd.DataFrame(short_answers.map(list).tolist(), index=exam.index,
                 columns=[f'B{i+1}' for i in range(N_B)]).eq('+')
B_shares = pd.DataFrame({'выполнено, %': B.mean().mul(100), 'не выполнено, %': (~B).mean().mul(100)})
display(B_shares)
assert np.allclose(B_shares.sum(axis=1), 100)

,"выполнено, %","не выполнено, %"
B1,89.261460,10.738540
B2,95.331070,4.668930
B3,92.784380,7.215620
B4,91.001698,8.998302
B5,88.497453,11.502547
B6,73.174873,26.825127
B7,37.478778,62.521222
B8,47.835314,52.164686
B9,91.850594,8.149406
B10,71.477080,28.522920


**Что проверить и объяснить:** Каждый столбец — задание, каждая строка — ученик. Знаменатель по каждому заданию — все ученики.

### ЕГЭ · задание 10

10. Аналогично и с типом С (ответы с развёрнутым ответом).

<details><summary>Подсказка — открой после своей попытки</summary>

У задания C бывают частичные баллы. Покажи три состояния: 0, частично, полный максимум.

</details>

In [74]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [75]:
C_points = pd.DataFrame([[int(pair[0]) for pair in row] for row in long_pairs], index=exam.index,
                        columns=[f'C{i+1}' for i in range(N_C)])
C_max = pd.DataFrame([[int(pair[1]) for pair in row] for row in long_pairs], index=exam.index,
                     columns=C_points.columns)
assert (C_max > 0).all().all() and ((C_points >= 0) & (C_points <= C_max)).all().all()
C_fraction = C_points / C_max
C_shares = pd.DataFrame({'нулевые баллы, %': C_points.eq(0).mean().mul(100),
                         'частично, %': ((C_points > 0) & (C_points < C_max)).mean().mul(100),
                         'полностью, %': C_points.eq(C_max).mean().mul(100)})
display(C_shares)
display(C_points.gt(0).mean().mul(100).rename('Получили хотя бы один балл, %'))
assert np.allclose(C_shares.sum(axis=1), 100)

,"нулевые баллы, %","частично, %","полностью, %"
C1,66.086587,7.767402,26.146010
C2,84.125637,8.404075,7.470289
C3,84.422750,3.013582,12.563667
C4,97.283531,2.207131,0.509338
C5,96.604414,2.292020,1.103565
C6,98.896435,0.721562,0.382003
C7,96.604414,3.353141,0.042445


C1    33.913413
C2    15.874363
C3    15.577250
C4     2.716469
C5     3.395586
C6     1.103565
C7     3.395586
Name: Получили хотя бы один балл, %, dtype: float64

**Что проверить и объяснить:** «Выполнено» неоднозначно для частичных баллов: здесь отдельно показаны полный ответ и хотя бы один балл.

### ЕГЭ · задание 11

11. Сделайте анализ по двум школам: по всем выполненным заданиям типа В; по заданиям типа С больше 50%; по среднему баллу юношей и девушек.

<details><summary>Подсказка — открой после своей попытки</summary>

Выбери две школы. Для C отдельно посчитай успех по каждому вопросу и по всей части, чтобы явно обозначить смысл порога.

</details>

In [76]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [77]:
SELECTED_SCHOOLS = exam['№ школы'].value_counts().head(2).index.tolist()
school_mask = exam['№ школы'].isin(SELECTED_SCHOOLS)
selected_exam = exam.loc[school_mask].copy()
selected_exam['Все B выполнены'] = B.loc[school_mask].all(axis=1)
selected_exam['Доля баллов C'] = (C_points.sum(axis=1) / C_max.sum(axis=1)).loc[school_mask]
selected_exam['C суммарно >50%'] = selected_exam['Доля баллов C'] > 0.5
school_summary = selected_exam.groupby('№ школы').agg(
    учеников=('Балл', 'size'), все_B=('Все B выполнены', 'sum'),
    все_B_доля=('Все B выполнены', 'mean'), C_выше_50=('C суммарно >50%', 'sum'),
    C_выше_50_доля=('C суммарно >50%', 'mean'))
display(school_summary)
display(B.loc[school_mask].groupby(exam.loc[school_mask, '№ школы']).mean().mul(100).round(1))
display(C_fraction.loc[school_mask].gt(0.5).groupby(exam.loc[school_mask, '№ школы']).mean().mul(100).round(1))
display(selected_exam.groupby(['№ школы', 'Пол'])['Балл'].agg(['size', 'mean']).round(2))

,учеников,все_B,все_B_доля,C_выше_50,C_выше_50_доля
№ школы,,,,,
135,182,18,0.098901,0,0.000000
136,1020,133,0.130392,13,0.012745


,B1,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12
№ школы,,,,,,,,,,,,
135,88.5,96.2,91.8,85.7,91.8,72.0,39.0,44.0,89.6,72.5,54.9,42.9
136,90.3,96.5,94.7,91.1,90.5,74.6,42.2,47.9,91.1,73.0,61.2,48.5


,C1,C2,C3,C4,C5,C6,C7
№ школы,,,,,,,
135,22.5,3.8,6.6,0.5,0.5,0.0,0.0
136,31.9,10.4,17.0,1.0,2.8,1.0,0.1


size   mean
№ школы Пол             
135     Ж      84  48.43
        М      98  47.04
136     Ж     532  50.98
        М     488  52.40

**Что проверить и объяснить:** Строго `>50%`, а не `>=50%`. Указаны и доля учеников с результатом выше 50% по всей части C, и доли по каждому заданию. Не смешивай долю учеников и долю баллов.

## Дополнительные файлы папки `02`

`sp500hst.txt` и `sp_data2.csv` — ещё два набора для практики чтения и соединения. В исходной лабораторной и PDF по ЕГЭ отдельных заданий для них нет; следующая тренировка добавлена специально, чтобы все табличные файлы папки были доступны в курсе. Это исторические учебные данные, не источник текущих котировок.

### Бонус: CSV без заголовка и разные разделители

Прочитай историю торгов (`sp500hst.txt`) с полями date, ticker, open, high, low, close, volume. Прочитай `sp_data2.csv` с разделителем `;` и полями ticker, company, weight. Соедини таблицы по тикеру и покажи первые строки. Не угадывай, что означает вес вне контекста источника.

<details><summary>Подсказка — открой после своей попытки</summary>

`header=None`, `names=[...]`; дату вида 20090821 преобразуй с `format="%Y%m%d"`. Для связи используй left-merge.

</details>

In [78]:
# Твоё решение
# Напиши код здесь. Исходные данные и примеры выше можно использовать.

**Эталон и разбор.** Сначала сравни идею решения со своей, затем запусти ячейку ниже.

In [79]:
prices = pd.read_csv(ROOT / '02/sp500hst.txt', header=None,
                     names=['date', 'ticker', 'open', 'high', 'low', 'close', 'volume'])
companies = pd.read_csv(ROOT / '02/sp_data2.csv', sep=';', header=None, names=['ticker', 'company', 'weight'])
prices['date'] = pd.to_datetime(prices['date'].astype(str), format='%Y%m%d')
assert companies['ticker'].is_unique
price_details = prices.merge(companies, on='ticker', how='left', validate='many_to_one')
display(price_details.head())
print('Строк:', len(price_details), 'Тикеров без справочника:', price_details.loc[price_details['company'].isna(), 'ticker'].nunique())

,date,ticker,open,high,low,close,volume,company,weight
0,2009-08-21,A,25.60,25.6100,25.220,25.55,34758,Agilent Technologies,0.1%
1,2009-08-24,A,25.64,25.7400,25.330,25.50,22247,Agilent Technologies,0.1%
2,2009-08-25,A,25.50,25.7000,25.225,25.34,30891,Agilent Technologies,0.1%
3,2009-08-26,A,25.32,25.6425,25.145,25.48,33334,Agilent Technologies,0.1%
4,2009-08-27,A,25.50,25.5700,25.230,25.54,70176,Agilent Technologies,0.1%


Строк: 122574 Тикеров без справочника: 182


**Что проверить и объяснить:** Left-merge сохраняет историю, даже если тикера нет в справочнике. Вес пока остаётся строкой с %: не усредняй его как число без определения.

## Дополнительная практика

Следующие блоки углубляют уже разобранные инструменты. Примеры с `sns.load_dataset` требуют интернета; задания с `train_test_split` — scikit-learn, с KDE — SciPy. Сначала мини-лекция по блоку, затем упражнения.

Пример данных:

```python
import pandas as pd
```

## Блок 1. Series

`Series(data,index,name,dtype)` — значения с метками. `loc` выбирает метку, `iloc` — позицию. Серия из словаря связывает ключи со значениями. `sort_values(ascending=False)` сортирует значения; `.head(n)` ограничивает результат.

**Пример записи**

```python
s = pd.Series([10,20],index=[1,3],name="score")
print(s.loc[1],s.iloc[1])
```

**Тренировка:** задания ниже используют эти инструменты.

### 1.1
sr = pd.Series([5, 6, 2, 9, 12], index=['a', 'b', 'c', 'd', 'e'])
Выведи: type(sr.values), sr.index, sr['b'], sr['b':'d'], sr.iloc[1:3].
Ожидание: numpy.ndarray; Index(['a','b','c','d','e']); 6;
  sr['b':'d'] -> три элемента b, c, d (правая граница ВКЛЮЧАЕТСЯ при явных метках);
  sr.iloc[1:3] -> два элемента b, c (при позиционных индексах граница НЕ включается).

In [80]:
# Дополнительная практика: твоё решение

### 1.2: Series из словаря
pop = pd.Series({'California': 38332521, 'Texas': 26448193, 'New York': 19651127,
                 'Florida': 19552860, 'Illinois': 12882135})
а) 'Texas' in pop                         -> True
б) pop[pop > 20_000_000]                  -> California, Texas
в) pop.sort_values()                      -> порядок: Illinois, Florida, New York, Texas, California
г) pop.sort_values(ascending=False).head(2).index.tolist() -> ['California', 'Texas']

In [81]:
# Дополнительная практика: твоё решение

### 1.3: ловушка целочисленного индекса
s = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])
Выведи: s[1], s.loc[1], s.iloc[1], s.loc[1:3], s.iloc[1:3].
Ожидание: 'a', 'a', 'b', ['a','b'] (метки 1 и 3, граница включается), ['b','c'] (позиции 1 и 2).
Поведение s[1:3] (без loc/iloc) зависит от версии pandas и неочевидно — не используй его.
Вывод: в рабочем коде всегда явно пиши .loc или .iloc.

In [82]:
# Дополнительная практика: твоё решение

## Блок 2. Создание DataFrame

`DataFrame(data,index,columns)` создаёт таблицу из словаря списков, массива или записей. Series при создании таблицы выравниваются по индексу. Если ключа в записи нет, появится пропуск. `df[column]` выбирает столбец, `df[[columns]]` — таблицу.

**Пример записи**

```python
df = pd.DataFrame([{"a":1},{"a":2,"b":3}])
print(df.dtypes)
```

**Тренировка:** задания ниже используют эти инструменты.

### 2.1: DataFrame из двух Series
pop — Series из 1.2, area = pd.Series({'California': 423967, 'Texas': 695662,
  'New York': 141297, 'Florida': 170312, 'Illinois': 149995})
states = pd.DataFrame({'population': pop, 'area': area})
Выведи shape, columns, index. Ожидание: (5, 2).
Добавь столбец density = population / area.
Ожидание: California 90.41, Texas 38.02, New York 139.08, Florida 114.81, Illinois 85.88.
Далее в файле states означает этот DataFrame (с density).

In [83]:
# Дополнительная практика: твоё решение

### 2.2: из словаря списков
data = {'county': ['Cochice', 'Pima', 'Santa Cruz', 'Maricopa', 'Yuma'],
        'year': [2012, 2012, 2013, 2014, 2014],
        'reports': [4, 24, 31, 2, 3]}
Создай DataFrame с порядком столбцов ['reports', 'county'] и индексом ['a', 'b', 'c', 'd', 'e'].
Ожидание: shape (5, 2), столбец year отсутствует. Как называется параметр для выбора столбцов?

In [84]:
# Дополнительная практика: твоё решение

### 2.3: пропуски при создании
rows = [{'a': 1, 'c': 'Alpha'}, {'a': 0, 'b': 3, 'c': 'Beta'}]
Создай DataFrame из rows и выведи dtypes.
Ожидание: столбцы a, c, b; b = [NaN, 3.0], dtype float64.
Объясни, почему b стал float64, хотя 3 — целое.

In [85]:
# Дополнительная практика: твоё решение

### 2.4: pandas против NumPy: что такое df[1]
np1 = np.array([[1, 2, 3], [4, 5, 6]])
а) np1[1]               -> [4 5 6]  (строка!)
б) df = pd.DataFrame(np1); df[1] -> столбец: 0 -> 2, 1 -> 5
в) df2 = pd.DataFrame(np1, index=['la', 'lb'], columns=['cl1', 'cl2', 'cl3']);
   df2['cl2'] -> la 2, lb 5;  df2['cl2']['la'] -> 2
Сформулируй правило: квадратные скобки у DataFrame по умолчанию выбирают столбцы.
Как получить строку 1 из df? (df.iloc[1])

In [86]:
# Дополнительная практика: твоё решение

## Блок 3. Индексация DataFrame

У `.loc[rows,columns]` метки, у `.iloc[rows,columns]` позиции. Срез loc включает правую метку. Для фильтрации комбинируй маски через `&` со скобками. Изменяй данные одним `.loc`, а для независимых операций делай `.copy()`.

**Пример записи**

```python
df.loc[df["a"] > 1, "b"] = 0
```

**Тренировка:** задания ниже используют эти инструменты.

### 3.1: столбцы
а) states['area'] — тип? (Series)    б) states.area — то же, но когда это ломается?
   (имя совпадает с методом или содержит пробел)
в) states[['population', 'area']] — shape (5, 2) и тип DataFrame

In [87]:
# Дополнительная практика: твоё решение

### 3.2: loc и iloc
а) states.loc['Texas', 'area']                 -> 695662
б) states.loc['Texas':'New York']              -> 2 строки (Texas, New York), граница включается
в) states.iloc[1:3]                            -> те же две строки
г) states.loc[:, 'population':'area']          -> два столбца
д) states.iloc[0, 2]                           -> density Калифорнии ≈ 90.41

In [88]:
# Дополнительная практика: твоё решение

### 3.3: маски
а) states[states['density'] > 100]                    -> New York, Florida
б) states.loc[states['density'] > 100, ['population', 'density']]  -> shape (2, 2)
в) states[(states['population'] > 20_000_000) & (states['density'] < 100)]
   Ожидание: California, Texas.  Почему здесь & и скобки, а не and?

In [89]:
# Дополнительная практика: твоё решение

### 3.4: присваивание и цепные индексы
а) states.loc['Illinois', 'density'] = 90 — правильное присваивание.
б) states[states['density'] > 100]['area'] = 0 — цепное присваивание.
   Выполни и проверь states: area не изменилась (или pandas выдал предупреждение).
   Объясни: первая скобка возвращает копию/временный объект, запись идёт в него.
в) Как правильно обнулить area у строк с density > 100? (через .loc с маской)

In [90]:
# Дополнительная практика: твоё решение

## Блок 4. Универсальные функции и выравнивание по индексу

Pandas выравнивает арифметику по меткам, а не позициям. При несовпавших индексах/столбцах возникает пропуск. Метод `.add(other,fill_value=0)` задаёт заполнение для отсутствующей стороны. NumPy-ufunc часто сохраняет метки Pandas.

**Пример записи**

```python
s1 = pd.Series([1,2],index=["a","b"])
s2 = pd.Series([10,20],index=["b","c"])
result = s1.add(s2,fill_value=0)
```

**Тренировка:** задания ниже используют эти инструменты.

### 4.1: ufunc сохраняют индекс
df = pd.DataFrame(np.arange(12).reshape(3, 4), columns=list('ABCD'))
np.sqrt(df) и df * 2 — убедись, что index и columns те же, исходный df не изменился.

In [91]:
# Дополнительная практика: твоё решение

### 4.2: выравнивание Series
s1 = pd.Series([1, 2, 3], index=['a', 'b', 'c'])
s2 = pd.Series([10, 20, 30], index=['b', 'c', 'd'])
а) s1 + s2                      -> a NaN, b 12, c 23, d NaN
б) s1.add(s2, fill_value=0)     -> a 1, b 12, c 23, d 30
Почему в NumPy складывались бы позиции, а в pandas — метки?

In [92]:
# Дополнительная практика: твоё решение

### 4.3: выравнивание DataFrame
dfa = pd.DataFrame({'A': [1, 2], 'B': [3, 4]}, index=[0, 1])
dfb = pd.DataFrame({'B': [10, 20, 30], 'C': [1, 1, 1]}, index=[1, 2, 3])
Предскажи, сколько значений в dfa + dfb НЕ NaN, потом проверь через .notna().sum().sum().
Ожидание: 1 (только строка 1, столбец B: 4 + 10 = 14). Результат содержит объединение индексов
и объединение столбцов.

In [93]:
# Дополнительная практика: твоё решение

## Блок 5. Пропуски

`isna` находит пропуски, `dropna(axis,how,thresh,subset)` удаляет неполные строки/столбцы, `fillna(value)` заполняет. `ffill` переносит последнее известное значение. Nullable-тип `Int64` хранит целые и `pd.NA`, обычный NumPy int — нет. Политику заполнения выбирают по смыслу данных.

**Пример записи**

```python
s = pd.Series([1,None,3],dtype="Int64")
filled = s.fillna(0)
```

**Тренировка:** задания ниже используют эти инструменты.

df = pd.DataFrame({'x': [1, np.nan, 3, np.nan],
                   'y': [np.nan, np.nan, 6, 7],
                   'z': [1, 2, 3, 4]})

In [94]:
# Дополнительная практика: твоё решение

### 5.1
а) df.isna().sum()      -> x 2, y 2, z 0
б) df.isna().mean()     -> x 0.5, y 0.5, z 0.0  (доля пропусков)
в) почему mean() от булевой маски даёт долю?

In [95]:
# Дополнительная практика: твоё решение

### 5.2: dropna
а) df.dropna()                  -> одна строка (индекс 2)
б) df.dropna(how='all')         -> все 4 строки
в) df.dropna(axis=1)            -> только столбец z
г) df.dropna(thresh=2)          -> строки 0, 2, 3 (минимум 2 непустых значения)

In [96]:
# Дополнительная практика: твоё решение

### 5.3: fillna
а) df.fillna(0)
б) df.fillna(df.mean())         -> x: [1, 2, 3, 2];  y: [6.5, 6.5, 6, 7]
в) df['x'].ffill()              -> [1, 1, 3, 3]
Методологическая заметка: б) на всём датасете перед split — утечка (среднее учитывает test).
В уровне 9 это делается правильно.

In [97]:
# Дополнительная практика: твоё решение

### 5.4: NaN и целые числа
pd.Series([1, 2, None]).dtype            -> float64
pd.Series([1, 2, None], dtype='Int64')   -> nullable-тип, пропуск показывается как <NA>
Объясни, почему обычный int-столбец не может хранить NaN.

In [98]:
# Дополнительная практика: твоё решение

## Блок 6. Агрегирование

Агрегаты `mean/sum/min/max/median` считают по столбцам при axis=0. `describe` собирает статистику, `quantile(q)` — квантили, `agg` — выбранные операции. `std(ddof=1)` Pandas отличается от NumPy с ddof=0: знаменатель n−1 вместо n.

**Пример записи**

```python
summary = df[["a","b"]].agg(["min","max","mean"])
quartiles = df[["a","b"]].quantile([0.25,0.5,0.75])
```

**Тренировка:** задания ниже используют эти инструменты.

df = pd.DataFrame({'A': [1, 2, 3, 4, 5], 'B': [10, 20, 30, 40, 50]})

### 6.1
а) df.mean()            -> A 3.0, B 30.0           (по умолчанию axis=0)
б) df.mean(axis=1)      -> [5.5, 11.0, 16.5, 22.0, 27.5]
в) df.sum(), df.median(), df.min(), df.max()

In [99]:
# Дополнительная практика: твоё решение

### 6.2: ddof — частая ошибка
df['A'].std()           -> 1.5811 (pandas: ddof=1, выборочное)
df['A'].values.std()    -> 1.4142 (NumPy: ddof=0, генеральное)
df['A'].values.std(ddof=1) совпадает с pandas.
Объясни разницу формул (деление на n против n-1). Какой вариант использует StandardScaler в sklearn?
(ddof=0.) Это важно, когда сверяешь свою стандартизацию с sklearn.

In [100]:
# Дополнительная практика: твоё решение

### 6.3: describe, quantile, agg
а) df.describe()                          — 8 строк статистик
б) df.quantile([0.25, 0.5, 0.75])         -> A: 2, 3, 4;  B: 20, 30, 40
в) df.agg(['min', 'max', 'mean'])         -> shape (3, 2)
г) df.agg({'A': 'sum', 'B': 'max'})       -> A 15, B 50

In [101]:
# Дополнительная практика: твоё решение

## Блок 7. Объединение таблиц

`concat(objects,axis=0,ignore_index=True)` складывает таблицы, а `merge` связывает по ключам. `on` — одинаковое имя, `left_on/right_on` — разные. `how` выбирает сохраняемые ключи. `validate` проверяет кратность; many-to-many может резко увеличить число строк.

**Пример записи**

```python
joined = left.merge(right,on="k",how="left",validate="one_to_one")
```

**Тренировка:** задания ниже используют эти инструменты.

### 7.1: concat
df1 = pd.DataFrame([[1, 2], [3, 4]], columns=list('AB'))
df2 = pd.DataFrame([[5, 6], [7, 8]], columns=list('AB'))
а) pd.concat([df1, df2]).index              -> [0, 1, 0, 1] (индекс дублируется)
б) pd.concat([df1, df2], ignore_index=True) -> индекс 0..3
в) pd.concat([df1, df2], axis=1).shape      -> (2, 4)
Метод df.append из лекции в современных версиях не существует.

In [102]:
# Дополнительная практика: твоё решение

### 7.2: данные для merge

In [103]:
# Дополнительная практика: твоё решение

Пример данных:

```python
employees = pd.DataFrame(
    {
        "employee": ["Bob", "Jake", "Lisa", "Sue"],
        "group": ["Accounting", "Engineering", "Engineering", "HR"],
    }
)
```

Пример данных:

```python
hire = pd.DataFrame(
    {"employee": ["Lisa", "Bob", "Jake", "Sue"], "hire_date": [2004, 2008, 2012, 2014]}
)
```

Пример данных:

```python
supervisors = pd.DataFrame(
    {
        "group": ["Accounting", "Engineering", "HR"],
        "supervisor": ["Carly", "Guido", "Steve"],
    }
)
```

Пример данных:

```python
skills = pd.DataFrame(
    {
        "group": ["Accounting", "Accounting", "Engineering", "Engineering", "HR", "HR"],
        "skills": [
            "math",
            "spreadsheets",
            "coding",
            "linux",
            "spreadsheets",
            "organization",
        ],
    }
)
```

а) один-к-одному: employees + hire по employee.
   Ожидание: 4 строки; Bob 2008, Jake 2012, Lisa 2004, Sue 2014.
б) многие-к-одному: результат (а) + supervisors по group.
   Ожидание: Bob -> Carly, Jake -> Guido, Lisa -> Guido, Sue -> Steve.
в) многие-ко-многим: employees + skills по group.
   Ожидание: shape (8, 3), у каждого сотрудника по 2 строки.
   Объясни, почему строк стало больше, чем в employees.

In [104]:
# Дополнительная практика: твоё решение

### 7.3: разные имена ключей

In [105]:
# Дополнительная практика: твоё решение

Пример данных:

```python
salary = pd.DataFrame(
    {"name": ["Bob", "Jake", "Lisa", "Sue"], "salary": [70000, 80000, 120000, 90000]}
)
```

Соедини employees и salary по employee == name (left_on / right_on),
затем удали лишний столбец name через drop(columns='name') БЕЗ inplace.
Ожидание: shape (4, 3): employee, group, salary.

In [106]:
# Дополнительная практика: твоё решение

### 7.4: тип соединения how

In [107]:
# Дополнительная практика: твоё решение

Пример данных:

```python
left = pd.DataFrame({"k": ["a", "b", "c"], "v1": [1, 2, 3]})
```

Пример данных:

```python
right = pd.DataFrame({"k": ["b", "c", "d"], "v2": [20, 30, 40]})
```

Выполни merge с how='inner', 'left', 'right', 'outer' и выведи число строк.
Ожидание: 2, 3, 3, 4. Для 'left' в строке a значение v2 = NaN.
Какой how по умолчанию? (inner)  Когда inner молча теряет данные?

In [108]:
# Дополнительная практика: твоё решение

## Блок 8. GroupBy: split -> apply -> combine

`groupby(by)` разбивает строки. `agg` сворачивает каждую группу, `transform` возвращает результат исходной длины для каждой строки, `filter` оставляет подходящие группы. `size` считает строки, `count` — непустые значения. `pivot_table(index,columns,values,aggfunc)` строит сводную.

**Пример записи**

```python
mean_in_group = sales.groupby("key")["data"].transform("mean")
centered = sales["data"] - mean_in_group
```

**Тренировка:** задания ниже используют эти инструменты.

Пример данных:

```python
sales = pd.DataFrame({"key": ["A", "B", "C", "A", "B", "C"], "data": range(1, 7)})
```

### 8.1
а) sales.groupby('key') — какой тип объекта и почему вычислений ещё нет?
б) sales.groupby('key').sum()    -> A 5, B 7, C 9
в) sales.groupby('key')['data'].mean()   -> A 2.5, B 3.5, C 4.5
г) sales.groupby('key').size()   -> по 2 в каждой группе

In [109]:
# Дополнительная практика: твоё решение

### 8.2: agg
sales.groupby('key')['data'].agg(['min', 'max', 'mean'])
Ожидание: A: 1, 4, 2.5;  B: 2, 5, 3.5;  C: 3, 6, 4.5

In [110]:
# Дополнительная практика: твоё решение

### 8.3: filter
Оставь только группы, у которых сумма data > 6.
Ожидание: группы B и C, shape (4, 2), исходные индексы сохранены (1, 2, 4, 5).

In [111]:
# Дополнительная практика: твоё решение

### 8.4: transform
Вычти из data среднее по группе (центрирование внутри группы).
Результат должен иметь ту же длину, что и sales, и тот же индекс.
Ожидание: [-1.5, -1.5, -1.5, 1.5, 1.5, 1.5]
Чем transform отличается от agg по форме результата?

In [112]:
# Дополнительная практика: твоё решение

### 8.5: несколько ключей и pivot

In [113]:
# Дополнительная практика: твоё решение

Пример данных:

```python
sales2 = pd.DataFrame(
    {
        "city": ["M", "M", "S", "S", "M", "S"],
        "product": ["x", "y", "x", "y", "x", "x"],
        "amount": [10, 20, 30, 40, 50, 60],
    }
)
```

а) sales2.groupby(['city', 'product'])['amount'].sum()  -> MultiIndex: M-x 60, M-y 20, S-x 90, S-y 40
б) .unstack()                                           -> таблица city x product
в) pd.pivot_table(sales2, values='amount', index='city', columns='product', aggfunc='sum')
   Ожидание в) и б): M: x=60, y=20;  S: x=90, y=40.

In [114]:
# Дополнительная практика: твоё решение

### 8.6: реальные данные, планеты
import seaborn as sns
planets = sns.load_dataset('planets')
а) planets.shape                                  -> (1035, 6)
б) число открытий каждым методом (planets.groupby('method').size());
   Ожидание: Radial Velocity 553, Transit 397.
в) медиана orbital_period по методам;  Radial Velocity 360.2, Transit ≈ 5.71.
г) сколько планет открыто в каждый год; год 2011 -> 185. Посмотри на какие годы пришёлся пик.
д) добавь столбец orbital_period_normalized = orbital_period / минимум orbital_period
   в рамках года — через groupby(...).transform('min'), а не через apply с изменением x
   (как в лекции: функция, мутирующая входную группу, — плохая практика).
е) что покажет count() для каждого столбца и чем он отличается от size()?
   (count не считает NaN, size — все строки)

In [115]:
# Дополнительная практика: твоё решение

## Блок 9. Мини-проект: подготовка Titanic без утечки данных

В подготовке модели сначала отделяй test. `train_test_split(test_size,stratify,random_state)` задаёт размер и воспроизводимость разбиения; stratify сохраняет доли классов. Медиана/мода и набор признаков рассчитываются по train. `get_dummies` создаёт признаки категорий, `reindex(columns=...,fill_value=0)` согласует test.

**Пример записи**

```python
age_median = train["age"].median()
test["age"] = test["age"].fillna(age_median)
test_encoded = test_encoded.reindex(columns=train_encoded.columns,fill_value=0)
```

**Тренировка:** задания ниже используют эти инструменты.

Тип задачи: бинарная классификация, цель survived.
Метрика (назови до работы с моделью и обоснуй): классы несбалансированы (≈38% / 62%),
поэтому одной accuracy мало — смотри ещё на F1 или ROC-AUC.
titanic = sns.load_dataset('titanic')

In [116]:
# Дополнительная практика: твоё решение

### 9.1: разведка
а) shape -> (891, 15)
б) число пропусков по столбцам: age 177, embarked 2, deck 688 (77% — столбец непригоден)
в) titanic['survived'].mean()  -> ≈ 0.384
г) доля выживших по sex:    female ≈ 0.74,  male ≈ 0.19
д) доля выживших по pclass: 1 ≈ 0.63,  2 ≈ 0.47,  3 ≈ 0.24
е) pivot_table по (sex, pclass) со средним survived
Запиши 2-3 вывода: какие признаки, судя по данным, самые информативные.

In [117]:
# Дополнительная практика: твоё решение

### 9.2: split и заполнение пропусков
1) Признаки: pclass, sex, age, sibsp, parch, fare, embarked. deck не берём (обоснуй).
2) ПЕРВЫМ шагом разбей на train/test: train_test_split(test_size=0.2,
   stratify=y, random_state=42). Почему stratify обязателен при дисбалансе классов?
   Ожидание: train 712 строк, test 179; доля survived в train и test ≈ 0.384 в обоих.
3) Медиану age считай ТОЛЬКО по train и заполни ею пропуски и в train, и в test.
4) Моду embarked считай ТОЛЬКО по train и заполни ею пропуски в обоих.
5) sex переведи в 0/1; embarked закодируй через pd.get_dummies на train, а для test сделай
   reindex(columns=train.columns, fill_value=0), чтобы набор столбцов совпал.
Проверки: нет NaN ни в train, ни в test; list(train.columns) == list(test.columns).
Письменно ответь: что именно утекло бы в test, если бы медиана age считалась до split
и по какой причине это завышает оценку качества модели.

In [118]:
# Дополнительная практика: твоё решение